# 06. 결측치 처리 비교 + 경쟁작 수 파생변수

**기준**: 04 최종 조합 = 베이스라인 + 데뷔작 처리 + 배급사 정리 + 개봉 시기 + 관람 등급 · sqrt 타깃 · TabPFN v3.5
(Public 1,229,915 · CV 1,357,181)

| 단계 | 내용 |
| --- | --- |
| 1 | `dir_prev_bfnum` 결측치 처리 3가지 비교: **0 + 데뷔작 플래그** (04 방식) / **0으로 일괄** / **결측 그대로 (TabPFN이 직접 처리)** |
| 2 | 1에서 가장 좋은 방식 위에 **경쟁작 수** 추가: `comp_all` / `comp_big` / 둘 다 |
| 3 | 최종 조합으로 학습 · 제출 파일 |

**경쟁작 수**: 개봉일 ±7일 안에 개봉한 다른 영화 수 (train + test 개봉일 사용, 관객수는 안 써서 누수 없음)
- `comp_all`: 전체 경쟁작 수 / `comp_big`: 그중 큰 영화(스태프 수 상위 25%) 수
- 기존 변수와 상관이 약하고 VIF 1.0~1.7 → 다중공선성 문제 없음 (2-1에서 확인)

- 검증: 팀 공통 5-Fold (seed 42), 원래 관객수 단위 RMSE · 결과는 `cache_06/`에 저장
- 파생변수 함수: `features.py`

## 0. 설정

In [1]:
import os
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn import set_config
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from features import make_features

warnings.filterwarnings('ignore')
set_config(display='text')

DATA_DIR = '.'
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE_DIR = 'cache_06'
os.makedirs(CACHE_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
y = train['box_off_num']
hit = y >= 1_000_000

VERSION = ModelVersion.V3_5   # 04와 같은 버전 (리더보드 최고)
TARGET_T = 'sqrt'
fwd, inv = np.sqrt, lambda p: np.square(np.clip(p, 0, None))
BASE = {'debut': 1, 'distributor': 1, 'date': 1, 'rating': 1}   # 04 최종 조합
print('device:', DEVICE)

device: cpu


C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. CV 함수

In [2]:
RESULTS = []


def make_model():
    return TabPFNRegressor.create_default_for_version(VERSION, device=DEVICE, random_state=SEED)


def run_cv(name, opts):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        res = pickle.load(open(path, 'rb'))
    else:
        X, _ = make_features(train, test, opts)
        oof = np.zeros(len(y))
        fold_rmse = []
        t0 = time.time()
        for tr_idx, va_idx in kf.split(X):
            model = make_model()
            model.fit(X.iloc[tr_idx], fwd(y.iloc[tr_idx]))
            oof[va_idx] = np.clip(inv(model.predict(X.iloc[va_idx])), 0, None)
            fold_rmse.append(mean_squared_error(y.iloc[va_idx], oof[va_idx]) ** 0.5)
        res = {'실험': name, '변수 수': X.shape[1],
               'CV RMSE': mean_squared_error(y, oof) ** 0.5,
               'fold 표준편차': np.std(fold_rmse),
               '흥행작 RMSE': mean_squared_error(y[hit], oof[hit]) ** 0.5,
               '소요(초)': time.time() - t0, 'opts': opts, 'oof': oof}
        pickle.dump(res, open(path, 'wb'))
    RESULTS.append(res)
    print(f"{name:<34} CV RMSE {res['CV RMSE']:>11,.0f} | fold std {res['fold 표준편차']:>9,.0f} | 흥행작 {res['흥행작 RMSE']:>11,.0f}")
    return res


FMT = {'CV RMSE': '{:,.0f}', 'fold 표준편차': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', '소요(초)': '{:.0f}'}

## 2. 결측치 처리 비교 (`dir_prev_bfnum`)
결측 330개(55%) = 이전 작품 수가 0인 영화와 정확히 일치 = **감독 데뷔작**

| 방식 | 처리 |
| --- | --- |
| `zero_flag` | 0으로 채우고 `is_debut` 플래그 추가 (04 방식) |
| `zero` | 0으로만 일괄 채우기 |
| `nan` | 결측 그대로 → TabPFN이 결측을 직접 처리 |

In [3]:
# 결측 확인
print('결측 개수  train:', train['dir_prev_bfnum'].isna().sum(), '/ test:', test['dir_prev_bfnum'].isna().sum())
print('결측 = 이전 작품 수 0 과 일치:', (train['dir_prev_bfnum'].isna() == (train['dir_prev_num'] == 0)).all())

miss = {}
for mode in ['zero_flag', 'zero', 'nan']:
    miss[mode] = run_cv(f'M_{mode}', {**BASE, 'debut_mode': mode})

best_mode = min(miss, key=lambda m: miss[m]['CV RMSE'])
print('\n→ 결측치 처리 채택:', best_mode)

결측 개수  train: 330 / test: 136
결측 = 이전 작품 수 0 과 일치: True
M_zero_flag                        CV RMSE   1,357,181 | fold std   320,882 | 흥행작   3,051,397
M_zero                             CV RMSE   1,354,539 | fold std   318,916 | 흥행작   3,040,116
M_nan                              CV RMSE   1,350,295 | fold std   314,851 | 흥행작   3,025,307

→ 결측치 처리 채택: nan


## 3. 경쟁작 수 추가
### 3-1. 경쟁작 수 확인 (분포 · 관객수와의 관계 · 다중공선성)

In [4]:
X_chk, _ = make_features(train, test, {**BASE, 'debut_mode': best_mode, 'comp_all': 1, 'comp_big': 1})
chk = X_chk[['comp_all', 'comp_big']].copy()
display(chk.describe().T.style.format('{:.1f}'))

print('관객수와의 Spearman 상관:')
for c in ['comp_all', 'comp_big']:
    print(f'  {c}: {chk[c].corr(y, method="spearman"):.3f}')

# 다중공선성: VIF (수치형 변수만, 5~10 넘으면 문제로 봄)
num = X_chk.loc[:, X_chk.nunique() > 2].astype(float)
num = num.fillna(0)   # VIF 계산용으로만 결측을 0으로 (모델 입력은 그대로)
num = (num - num.mean()) / num.std()
vif = pd.Series(np.diag(np.linalg.pinv(np.corrcoef(num.values.T))), index=num.columns).sort_values(ascending=False)
display(vif.to_frame('VIF').style.format('{:.2f}'))

,count,mean,std,min,25%,50%,75%,max
comp_all,600.0,7.6,3.2,0.0,5.0,7.0,10.0,15.0
comp_big,600.0,1.7,1.2,0.0,1.0,1.0,2.0,6.0


관객수와의 Spearman 상관:
  comp_all: -0.182
  comp_big: -0.161


,VIF
월,19.28
quarter,19.07
staff_x_rat2,6.05
staff_x_rat3,4.63
num_staff,4.02
staff_x_rat1,3.82
staff_x_rat0,2.13
time,1.76
comp_all,1.59
dir_prev_num,1.45


### 3-2. 경쟁작 수 넣어보기
`comp_all` / `comp_big` / 둘 다 → 결측치 처리 최고 방식보다 CV가 좋아지면 채택

In [5]:
base_opts = {**BASE, 'debut_mode': best_mode}
best = miss[best_mode]
best_opts = base_opts

comp = {}
for name, extra in [('comp_all', {'comp_all': 1}), ('comp_big', {'comp_big': 1}), ('comp_both', {'comp_all': 1, 'comp_big': 1})]:
    comp[name] = run_cv(f'C_{best_mode}+{name}', {**base_opts, **extra})
    if comp[name]['CV RMSE'] < best['CV RMSE']:
        best, best_opts = comp[name], {**base_opts, **extra}

print('\n→ 최종 조합:', best['실험'], f"(CV RMSE {best['CV RMSE']:,.0f})")

C_nan+comp_all                     CV RMSE   1,368,135 | fold std   317,550 | 흥행작   3,072,463


C_nan+comp_big                     CV RMSE   1,353,551 | fold std   326,712 | 흥행작   3,051,695


C_nan+comp_both                    CV RMSE   1,350,817 | fold std   321,902 | 흥행작   3,037,640

→ 최종 조합: M_nan (CV RMSE 1,350,295)


## 4. 전체 결과

In [6]:
ref = 1_357_181   # 04 최종 CV RMSE
table = pd.DataFrame([{k: v for k, v in r.items() if k not in ('opts', 'oof')} for r in RESULTS]).drop_duplicates('실험')
table['04 대비'] = table['CV RMSE'] - ref
display(table.style.format({**FMT, '04 대비': '{:+,.0f}'}).highlight_min(subset=['CV RMSE'], color='#c6efce'))
table.to_csv('06_results.csv', index=False, encoding='utf-8-sig')
print('06_results.csv 저장 (노션 리더보드 기록용)')

,실험,변수 수,CV RMSE,fold 표준편차,흥행작 RMSE,소요(초),04 대비
0,M_zero_flag,42,"1,357,181","320,882","3,051,397",446,-0
1,M_zero,41,"1,354,539","318,916","3,040,116",446,"-2,642"
2,M_nan,41,"1,350,295","314,851","3,025,307",365,"-6,886"
3,C_nan+comp_all,42,"1,368,135","317,550","3,072,463",118,"+10,954"
4,C_nan+comp_big,42,"1,353,551","326,712","3,051,695",99,"-3,630"
5,C_nan+comp_both,43,"1,350,817","321,902","3,037,640",132,"-6,364"


06_results.csv 저장 (노션 리더보드 기록용)


## 5. 최종 학습 · 제출 파일

In [7]:
X, X_test = make_features(train, test, best_opts)
model = make_model()
model.fit(X, fwd(y))
pred = np.clip(inv(model.predict(X_test)), 0, None)

submission['box_off_num'] = pred
fname = f"06_{best['실험']}.csv"
submission.to_csv(fname, index=False)
print(f'저장: {fname} | 변수 {X.shape[1]}개 | 예측 평균 {pred.mean():,.0f} · 최대 {pred.max():,.0f}')

저장: 06_M_nan.csv | 변수 41개 | 예측 평균 636,347 · 최대 8,560,683
